# 02. RLHF Reward Modeling and PPO vs RLOO

- **Category:** 08_learning_paradigms_rl_and_alignment / 03_llm_alignment_and_reasoning
- **Lineage & Citations:** Bradley & Terry (1952); Christiano et al. (2017); Ziegler et al. (2019); Ouyang et al. (2022 InstructGPT); Ahmadian et al. (2024 REINFORCE Leave-One-Out / RLOO).
- **Purpose:** First-principles mathematical implementation and empirical verification of reinforcement learning from human feedback: Bradley-Terry preference reward modeling, PPO token-level reward shaping with reference KL regularization, and REINFORCE Leave-One-Out (RLOO) advantage estimation.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [01. Supervised Finetuning Chat Masks Neftune and Sequence Packing](01_supervised_finetuning_chat_masks_neftune_and_sequence_packing.ipynb) $\to$ `🔗 Downstream Syntheses:` [03. Direct Preference DPO IPO KTO SimPO DNO and SPO](03_direct_preference_dpo_ipo_kto_simpo_dno_and_spo.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to Use**  
> Never omit the reference KL penalty $\beta$ in RLHF; without $\beta$, the policy rapidly diverges into catastrophic reward-hacking gibberish. Do NOT attempt RLOO with $K=1$, as leave-one-out baseline estimation requires at least $K \ge 2$ sampled completions per prompt.

## Mathematical Foundations & Governing Equations

### 1. Bradley-Terry Preference Reward Loss
$$
\mathcal{L}_{\text{RM}}(\theta) = -\mathbb{E}_{(x, y_w, y_l)} \left[ \ln \sigma(r_\theta(x, y_w) - r_\theta(x, y_l)) \right] = \mathbb{E} \left[ \text{softplus}\left( -(r_\theta(x, y_w) - r_\theta(x, y_l)) \right) \right]
$$

### 2. RLHF Token-Level Reward Shaping
$$
r_t = \begin{cases} -\beta \left( \ln \pi_\theta(y_t \mid y_{<t}) - \ln \pi_{\text{ref}}(y_t \mid y_{<t}) \right) & \text{if } t < T \\ r_\phi(x, y) - \beta \left( \ln \pi_\theta(y_T \mid y_{<T}) - \ln \pi_{\text{ref}}(y_T \mid y_{<T}) \right) & \text{if } t = T \end{cases}
$$

### 3. RLOO Leave-One-Out Advantage
$$
b_i = \frac{1}{K - 1} \sum_{j \ne i} R_j = \frac{\left(\sum_{j=1}^K R_j\right) - R_i}{K - 1}
$$
$$
\hat{A}_i = R_i - b_i, \quad \text{with } \sum_{i=1}^K \hat{A}_i = 0
$$

### 4. RLOO Policy Gradient Objective
$$
\mathcal{L}_{\text{RLOO}}(\theta) = -\frac{1}{B \cdot K} \sum_{b=1}^B \sum_{k=1}^K \hat{A}_{b, k} \sum_{t=1}^{|y_{b, k}|} \ln \pi_\theta(y_{b, k, t} \mid y_{b, k, <t})
$$

### Asymptotic Time & Memory Complexity
- Bradley-Terry Loss: $\mathcal{O}(B)$ time and memory.
- Token-Level Reward Trajectory: $\mathcal{O}(B \cdot T)$ time and memory.
- RLOO Leave-One-Out Advantage: $\mathcal{O}(B \cdot K)$ time and memory.
- RLOO Policy Gradient Loss: $\mathcal{O}(B \cdot K)$ time and memory.

In [ ]:
# Setup and seed verification environment
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)

# Locate the golden checkpoint
CKPT_REL = Path("weights/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/02_rlhf_reward_modeling_and_ppo_vs_rloo.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x: torch.Tensor, w: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    return x @ w + b

def bradley_terry_reward_loss(
    rewards_chosen: torch.Tensor,
    rewards_rejected: torch.Tensor,
) -> torch.Tensor:
    diff = rewards_chosen - rewards_rejected
    return torch.mean(F.softplus(-diff))

def rlhf_token_level_rewards(
    log_probs_policy: torch.Tensor,
    log_probs_ref: torch.Tensor,
    terminal_scores: torch.Tensor,
    kl_coeff: float = 0.05,
) -> torch.Tensor:
    token_kl = log_probs_policy - log_probs_ref
    rewards = -kl_coeff * token_kl
    rewards[:, -1] += terminal_scores
    return rewards

def rloo_leave_one_out_advantages(
    scores: torch.Tensor,
) -> torch.Tensor:
    b, k = scores.shape
    total_scores = torch.sum(scores, dim=-1, keepdim=True)
    baselines = (total_scores - scores) / float(k - 1)
    return scores - baselines

def rloo_policy_gradient_loss(
    advantages: torch.Tensor,
    log_probs_sum: torch.Tensor,
) -> torch.Tensor:
    return -torch.mean(advantages * log_probs_sum)

In [ ]:
# Multi-track numerical parity assertions
y = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
err_anchor = float(torch.max(torch.abs(y - weights["test_output"])))
assert err_anchor < 1e-5, f"Linear anchor parity error: {err_anchor:.2e}"

bt_loss = bradley_terry_reward_loss(weights["rewards_chosen"], weights["rewards_rejected"])
err_bt = float(torch.abs(bt_loss - weights["bt_loss_ref"]))
assert err_bt < 1e-5, f"Bradley-Terry reward loss parity error: {err_bt:.2e}"

rew = rlhf_token_level_rewards(weights["log_probs_policy"], weights["log_probs_ref"], weights["terminal_scores"], kl_coeff=0.05)
err_rew = float(torch.max(torch.abs(rew - weights["rlhf_rewards_ref"])))
assert err_rew < 1e-5, f"RLHF token rewards parity error: {err_rew:.2e}"

adv = rloo_leave_one_out_advantages(weights["scores"])
err_adv = float(torch.max(torch.abs(adv - weights["rloo_adv_ref"])))
assert err_adv < 1e-5, f"RLOO advantages parity error: {err_adv:.2e}"

loss = rloo_policy_gradient_loss(weights["rloo_adv_ref"], weights["log_probs_sum"])
err_loss = float(torch.abs(loss - weights["rloo_loss_ref"]))
assert err_loss < 1e-5, f"RLOO policy gradient loss parity error: {err_loss:.2e}"

print(f"All parity assertions passed successfully (max error {max(err_anchor, err_bt, err_rew, err_adv, err_loss):.2e})")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

b_bt = benchmark_fn(
    lambda: bradley_terry_reward_loss(weights["rewards_chosen"], weights["rewards_rejected"]),
    iters=100,
    name="Bradley-Terry Reward Loss",
)
b_rew = benchmark_fn(
    lambda: rlhf_token_level_rewards(weights["log_probs_policy"], weights["log_probs_ref"], weights["terminal_scores"]),
    iters=100,
    name="RLHF Token Rewards Shaping",
)
b_adv = benchmark_fn(
    lambda: rloo_leave_one_out_advantages(weights["scores"]),
    iters=100,
    name="RLOO Leave-One-Out Advantages",
)
b_loss = benchmark_fn(
    lambda: rloo_policy_gradient_loss(weights["rloo_adv_ref"], weights["log_probs_sum"]),
    iters=100,
    name="RLOO Policy Gradient Loss",
)

print(format_benchmark_table([b_bt, b_rew, b_adv, b_loss]))

In [ ]:
# Failure Mode Contrast: Unbiased Zero-Sum Advantage in RLOO vs Non-Zero Bias in Vanilla REINFORCE
# Suppose 3 completions for a prompt receive positive scores: [2.0, 4.0, 6.0]
scores_toy = torch.tensor([[2.0, 4.0, 6.0]])

# Case 1: Vanilla REINFORCE without baseline
# All advantages are strictly positive (+2.0, +4.0, +6.0), reinforcing even below-average completions
vanilla_advantages = scores_toy
vanilla_sum = torch.sum(vanilla_advantages).item()
vanilla_min = torch.min(vanilla_advantages).item()

# Case 2: RLOO Leave-One-Out baseline
# Baseline for item 0: (4 + 6) / 2 = 5.0 -> Advantage = 2.0 - 5.0 = -3.0 (penalized!)
# Baseline for item 1: (2 + 6) / 2 = 4.0 -> Advantage = 4.0 - 4.0 =  0.0 (neutral!)
# Baseline for item 2: (2 + 4) / 2 = 3.0 -> Advantage = 6.0 - 3.0 = +3.0 (reinforced!)
rloo_adv = rloo_leave_one_out_advantages(scores_toy)
rloo_sum = torch.sum(rloo_adv).item()

print(f"Vanilla REINFORCE Minimum Advantage: {vanilla_min:.4f} (All positive! Reinforces mediocre outputs!)")
print(f"RLOO Advantages:                      {rloo_adv.numpy().tolist()[0]} (Zero-sum! Penalizes below-average!)")
print(f"RLOO Advantage Sum:                   {rloo_sum:.4f} (Strictly zero!)")

assert vanilla_min > 0.0, f"Vanilla advantages should be strictly positive (got {vanilla_min})"
assert abs(rloo_sum) < 1e-5, f"RLOO advantage sum must be exactly zero (got {rloo_sum})"
print("Pathological failure mode contrast successfully verified")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Bradley-Terry Formulation**: Cross-entropy over logistic scalar differences $\sigma(r_w - r_l)$ smoothly translates binary human preferences into continuous scalar utility values.
2. **Reference KL Anchor**: Penalizing per-token divergence prevents the policy from drifting out-of-distribution into regions where the reward model is uncalibrated.
3. **Critic-Free Advantage Estimation**: RLOO proves that a separate learned value network $V_\psi$ is unnecessary in post-training LLM alignment, halving GPU memory requirements.
4. **Exact Zero-Sum Centering**: Leave-One-Out advantages strictly sum to zero per prompt, guaranteeing unbiased relative reinforcement across parallel sample completions.

---
### Cross-Domain Generalisation Matrix
- **Dialogue Safety Alignment**: Bradley-Terry reward models filter out toxic, harmful, or legally prohibited completions.
- **Code Generation Pass@k Optimization**: Unit-test execution rewards train programming policies via RLOO without critic instability.
- **Creative Writing Persona Tuning**: Style and tone preference rewards guide creative story generation while reference KL preserves grammatical fluency.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [02_rlhf_reward_modeling_and_ppo_vs_rloo.md](../../../markdown/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/02_rlhf_reward_modeling_and_ppo_vs_rloo.md)
- **PyTorch 2.x Module:** [02_rlhf_reward_modeling_and_ppo_vs_rloo.py](../../../pytorch2/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/02_rlhf_reward_modeling_and_ppo_vs_rloo.py)
- **Functional JAX Module:** [02_rlhf_reward_modeling_and_ppo_vs_rloo.py](../../../jax/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/02_rlhf_reward_modeling_and_ppo_vs_rloo.py)
- **Native Rust Candle Track:** [02_rlhf_reward_modeling_and_ppo_vs_rloo.rs](../../../candle/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/02_rlhf_reward_modeling_and_ppo_vs_rloo.rs)
- **Hardware Kernel (CPU / WGSL):** [02_rlhf_reward_modeling_and_ppo_vs_rloo.rs](../../../kernels/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/02_rlhf_reward_modeling_and_ppo_vs_rloo.rs)
- **Golden SafeTensors:** [02_rlhf_reward_modeling_and_ppo_vs_rloo.safetensors](../../../weights/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/02_rlhf_reward_modeling_and_ppo_vs_rloo.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 01. Supervised Finetuning Chat Masks Neftune and Sequence Packing](01_supervised_finetuning_chat_masks_neftune_and_sequence_packing.ipynb) | [Master Index](../../../README.md#08_learning_paradigms_rl_and_alignment) | [Next: 03. Direct Preference DPO IPO KTO SimPO DNO and SPO](03_direct_preference_dpo_ipo_kto_simpo_dno_and_spo.ipynb) &rarr;